 # V4 Prototypical Network (DEFINITIVE & EXHAUSTIVE)

 This notebook integrates EVERY parameter, augmentation, backbone, and preprocessing step.

In [1]:
import os, sys, warnings, logging, numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.spatial.transform import Rotation as R
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit, train_test_split
from sklearn.metrics import f1_score, make_scorer

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)
src_path = os.path.join(workspace_root, "src")
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

try:
    import data_utils
    from base_utils_qwen import prepare_bayesian_space, competition_scorer as bfrb_competition_scorer, evaluate_holdout, SequenceExtractor
    from proto_utils_v4 import V4PrototypicalNetwork, V4MultiHeadPrototypicalNetwork
    print("✅ Imports loaded successfully")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    raise


✅ Imports loaded successfully


 # 1. Global Configuration & Slicing Flags

In [2]:
mode = "single"  # "single" or "multi"
single_target = "bfrb" 
target_col = single_target

multi_heads = ["gesture_action", "orientation", "gesture_position"]
primary_target = "bfrb" 

pipe_name = "extractor"
proto_name = "model"

search_mode = "grid"  # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 30  
n_jobs = 1
train_size = 0.5
error_score_constant = 'raise'
verbose = 3
do_cross_val = False

if do_cross_val: cv_object = GroupKFold(n_splits=n_splits)
else: cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

if target_col == 'bfrb': competition_scorer = bfrb_competition_scorer
else: competition_scorer = make_scorer(f1_score, average="macro", zero_division=0)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# DATA FILTERING & SLICING FLAGS (From siamese/prototypical_v2)
use_eg_sample = False
eg_sample_pct = 0.02
do_handness = False                 
do_upside_down = False
filter_non_bfrb_classes = False    
filter_orientation_class_list = None 
slice_by_orientation = None        
slice_by_bfrb = None               

print(f"Mode: {mode} | Search: {search_mode} | Target: {target_col}")


Mode: single | Search: grid | Target: bfrb


 # 2. Data Loading & Exhaustive Preprocessing

In [3]:
data_root = data_utils.find_data_root()
eg_path = data_root / "eg.csv"

if use_eg_sample and eg_path.exists():
    raw_train_df = pd.read_csv(eg_path)
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
else:
    raw_train_df = pd.read_csv(data_root / "train.csv")
    train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
    if use_eg_sample:
        seq_ids = raw_train_df["sequence_id"].drop_duplicates().sample(frac=eg_sample_pct, random_state=random_state)
        raw_train_df = raw_train_df[raw_train_df["sequence_id"].isin(seq_ids)].copy()

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if do_handness and "handedness" in train_demo_df.columns:
    train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
    left_handed_mask = train_df["handedness"].eq(0)
    train_df.loc[left_handed_mask, "acc_x"] *= -1.0
    rot_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]
    q_wxyz = train_df.loc[left_handed_mask, rot_cols].to_numpy(dtype=float)
    q_wxyz = np.nan_to_num(q_wxyz, nan=0.0, posinf=0.0, neginf=0.0)
    norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
    bad_q = norm.squeeze() == 0
    q_wxyz[bad_q] = np.array([1.0, 0.0, 0.0, 0.0])
    norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
    q_wxyz = q_wxyz / norm
    q_xyzw = q_wxyz[:, [1, 2, 3, 0]]
    euler_xyz = R.from_quat(q_xyzw).as_euler("xyz", degrees=False)
    euler_xyz[:, [1, 2]] *= -1.0
    q_xyzw_fixed = R.from_euler("xyz", euler_xyz, degrees=False).as_quat()
    q_wxyz_fixed = q_xyzw_fixed[:, [3, 0, 1, 2]]
    train_df.loc[left_handed_mask, rot_cols] = q_wxyz_fixed
    train_df = train_df.drop(columns=["handedness"])

if do_upside_down:
    upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
    train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
    train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df['is_target'] = (train_df['sequence_type'] == 'Target').astype(bool)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'], 'non_bfrb')

if filter_non_bfrb_classes: train_df = train_df.loc[train_df['is_target']]
if filter_orientation_class_list is not None: train_df = train_df[train_df['orientation'].isin(filter_orientation_class_list)]
if slice_by_bfrb is True: train_df = train_df.loc[train_df['is_target']]
elif slice_by_bfrb is False: train_df = train_df.loc[~train_df['is_target']]
if slice_by_orientation is not None: train_df = train_df[train_df['orientation'].isin(slice_by_orientation)]

if filter_orientation_class_list is not None or slice_by_orientation is not None:
    sequences = train_df[['sequence_id', 'is_target', 'bfrb', 'orientation', 'gesture_action']].drop_duplicates()
    train_seqs, test_seqs = train_test_split(sequences['sequence_id'], test_size=(1 - train_size), stratify=sequences[target_col], random_state=random_state)
    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy()
else:
    try:
        train_sample_df, hold_out_df = data_utils.sample_balanced_split(train_df, train_pct=train_size, test_pct=min(0.2, 1 - train_size), random_state=random_state)
    except Exception:
        unique_seqs = train_df[["sequence_id", target_col]].drop_duplicates("sequence_id").sample(frac=1, random_state=random_state)
        n_train = int(len(unique_seqs) * train_size)
        train_seqs = unique_seqs.iloc[:n_train]["sequence_id"]
        test_seqs = unique_seqs.iloc[n_train:]["sequence_id"]
        train_sample_df = train_df[train_df["sequence_id"].isin(train_seqs)].copy()
        hold_out_df = train_df[train_df["sequence_id"].isin(test_seqs)].copy()

X_train, X_test = train_sample_df.copy(), hold_out_df.copy()
if mode == "single":
    y_train = train_sample_df[["sequence_id", "is_target", single_target]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", single_target]].copy()
else:
    cols = ["sequence_id"] + multi_heads
    if primary_target not in cols: cols.append(primary_target)
    y_train = train_sample_df[cols].copy()
    y_test = hold_out_df[["sequence_id", "is_target", primary_target]].copy()

groups = X_train["sequence_id"]
print(f"✅ Split Complete: Train={X_train['sequence_id'].nunique()} seqs | Test={X_test['sequence_id'].nunique()} seqs")


✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Train: 2910 seqs | 35.7%
Test:  969 seqs  | 11.9%
✅ Split Complete: Train=2910 seqs | Test=969 seqs


 # 3. Pipeline Initialization

In [4]:
sequence_extractor = SequenceExtractor()

if mode == "single":
    model = V4PrototypicalNetwork(target=single_target)
else:
    model = V4MultiHeadPrototypicalNetwork(primary_target=primary_target, sub_heads=multi_heads)

pipeline = Pipeline([
    (pipe_name, sequence_extractor),
    (proto_name, model),
])


 # 4. EXHAUSTIVE Bayesian Parameter Space

In [5]:
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    param_space = {
        # EXTRACTOR (base_utils_qwen exact match)
        f"{pipe_name}__acc_modes": Categorical(["raw", "smoothed|velocity|jerk", "raw|velocity|displacement|jerk", "smoothed"]),
        f"{pipe_name}__rotation_modes": Categorical(["quaternion", "quaternion|angular_velocity", "quaternion|euler|angular_velocity", "quaternion|delta_euler", "rot6d"]),
        f"{pipe_name}__tof_modes": Categorical(["sensor_stats", "pooled_stats", "sensor_stats|pooled", "raw"]),
        f"{pipe_name}__thm_modes": Categorical(["centered_diff", "diff", "centered", "raw"]),
        f"{pipe_name}__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-3, 10.0, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True, False]),
        f"{pipe_name}__dead_reckoning_detrend": Categorical([True, False]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__window_size": Integer(3, 50),
        f"{pipe_name}__smooth_alpha": Real(0.1, 0.9),
        f"{pipe_name}__clip_value": Categorical([None, 50.0, 100.0]),
        f"{pipe_name}__interp_mode": Categorical(["linear", "ffill"]),
        f"{pipe_name}__maxlen": Categorical([120, 160, 200, 256]),
        f"{pipe_name}__padding_value": Categorical([-999.0, 0.0]),

        # --- IMU (Accelerometer) Sampling Rates ---
        f"{pipe_name}__imu_native_sampling_rate": Categorical([20, 50, 100]),
        f"{pipe_name}__imu_target_sampling_rate": Categorical([20, 50, 100]),
        
        # --- Rotation (Quaternion/Gyro) Sampling Rates ---
        f"{pipe_name}__rot_native_sampling_rate": Categorical([20, 50, 100]),
        f"{pipe_name}__rot_target_sampling_rate": Categorical([20, 50, 100]),
        
        # --- Time-of-Flight (TOF) Sampling Rates (Typically lower) ---
        f"{pipe_name}__tof_native_sampling_rate": Categorical([5, 10, 20]),
        f"{pipe_name}__tof_target_sampling_rate": Categorical([5, 10, 20]),
        
        # --- Thermopile (THM) Sampling Rates (Typically lower) ---
        f"{pipe_name}__thm_native_sampling_rate": Categorical([5, 10, 20]),
        f"{pipe_name}__thm_target_sampling_rate": Categorical([5, 10, 20]),
        f"{pipe_name}__chunk_window_size": Integer(20, 128),
        f"{pipe_name}__chunk_stride": Integer(10, 64),

        # MODEL BACKBONE & REGULARIZATION
        f"{proto_name}__backbone_type": Categorical(["1dcnn", "2dcnn", "lstm", "attention", "conv4", "resnet12", "resnet18"]),
        f"{proto_name}__filters": Categorical(["32-64", "64-128", "64-128-256", "128-256-256-256", "32-64-128"]),
        f"{proto_name}__kernels": Categorical(["3-3", "5-3", "7-5-3", "5-5-5-5", "3-3-3-3", "9-9-9-9"]),
        f"{proto_name}__pools": Categorical(["none", "2", "2-2", "2-2-2"]),
        f"{proto_name}__lstm_units": Categorical([64, 128, 256]),
        f"{proto_name}__attention_heads": Categorical([2, 4, 8]),
        f"{proto_name}__embed_dim": Categorical([32, 64, 128, 256]),
        f"{proto_name}__dropout": Categorical([0.0]),
        f"{proto_name}__spatial_dropout": Categorical([0.0]),
        f"{proto_name}__l2_reg": Categorical([1e-5]),
        f"{proto_name}__learning_rate": Categorical([5e-4]),
        f"{proto_name}__batch_size": Categorical([32]),
        f"{proto_name}__epochs": Categorical([50]),
        f"{proto_name}__patience": Categorical([10]),
        f"{proto_name}__validation_split": Categorical([0.1]),
        f"{proto_name}__temperature": Real(0.01, 0.5, prior="log-uniform"),
        f"{proto_name}__supcon_weight": Real(0.0, 0.5),
        f"{proto_name}__use_phase_attention": Categorical([True, False]),
        f"{proto_name}__modality_dropout_prob": Categorical([0.0, 0.3, 0.5]),
        f"{proto_name}__use_attention_prototypes": Categorical([True, False]),
        f"{proto_name}__n_way": Categorical([9]),
        f"{proto_name}__n_support": Categorical([5, 40]),
        f"{proto_name}__n_query": Categorical([5, 40]),

        # ALL 11 TEMPORAL AUGMENTATIONS
        f"{proto_name}__use_mixup": Categorical([True, False]),
        f"{proto_name}__mixup_alpha": Real(0.1, 0.8),
        f"{proto_name}__mixup_prob": Real(0.1, 0.9),
        f"{proto_name}__use_time_shift": Categorical([True, False]),
        f"{proto_name}__max_shift_pct": Real(0.05, 0.25),
        f"{proto_name}__use_time_stretch": Categorical([True, False]),
        f"{proto_name}__time_stretch_min": Real(0.7, 0.9),
        f"{proto_name}__time_stretch_max": Real(1.1, 1.3),
        f"{proto_name}__use_gaussian_noise": Categorical([True, False]),
        f"{proto_name}__noise_std": Real(1e-4, 0.05, prior="log-uniform"),
        f"{proto_name}__use_magnitude_scaling": Categorical([True, False]),
        f"{proto_name}__mag_min": Real(0.8, 0.95),
        f"{proto_name}__mag_max": Real(1.05, 1.2),
        f"{proto_name}__use_time_mask": Categorical([True, False]),
        f"{proto_name}__time_mask_ratio": Real(0.05, 0.3),
        f"{proto_name}__use_channel_dropout": Categorical([True, False]),
        f"{proto_name}__channel_drop_prob": Real(0.05, 0.3),
        f"{proto_name}__use_quaternion_flip": Categorical([True, False]),
        f"{proto_name}__quat_flip_prob": Real(0.1, 0.9),
        f"{proto_name}__use_freq_filter": Categorical([True, False]),
        f"{proto_name}__freq_keep_low": Real(0.05, 0.2),
        f"{proto_name}__freq_keep_high": Real(0.8, 0.95),
    }

    if mode == "multi":
        param_space[f"{proto_name}__uncertainty_weighting"] = Categorical([True, False])
        param_space[f"{proto_name}__primary_target"] = Categorical([primary_target])
        param_space[f"{proto_name}__sub_heads"] = Categorical([tuple(multi_heads)])
    else:
        param_space[f"{proto_name}__target"] = Categorical([single_target])

    param_space = prepare_bayesian_space(param_space)

else:
    param_space = {
        f"{pipe_name}__acc_modes": ["raw"],
        f"{pipe_name}__rotation_modes": ["raw"],
        f"{pipe_name}__tof_modes": ["sensor_stats"],
        f"{pipe_name}__thm_modes": ["centered_diff"],
        f"{pipe_name}__motion_filter_mode": [None],

        # --- SAMPLING RATES (Fixed to prevent combinatorial explosion) ---
        f"{pipe_name}__imu_native_sampling_rate": [100],
        f"{pipe_name}__imu_target_sampling_rate": [100],
        f"{pipe_name}__rot_native_sampling_rate": [20],
        f"{pipe_name}__rot_target_sampling_rate": [20],
        f"{pipe_name}__tof_native_sampling_rate": [5],
        f"{pipe_name}__tof_target_sampling_rate": [5],
        f"{pipe_name}__thm_native_sampling_rate": [5],
        f"{pipe_name}__thm_target_sampling_rate": [5],

        f"{pipe_name}__chunk_window_size": [64],
        f"{pipe_name}__chunk_stride": [32],
        f"{pipe_name}__padding_value": [0.0],
        f"{proto_name}__backbone_type": ["conv4", '1dcnn', '2dcnn', 'attention', 'lstm', 'resnet12', 'resnet18'], 
        f"{proto_name}__filters": ["64-128-256"],
        f"{proto_name}__kernels": ["3-3-3"],
        f"{proto_name}__pools": ["2"],
        f"{proto_name}__embed_dim": [128],
        f"{proto_name}__dropout": [0.0],
        f"{proto_name}__spatial_dropout": [0.0],
        f"{proto_name}__l2_reg": [1e-4],
        f"{proto_name}__learning_rate": [5e-4],
        f"{proto_name}__batch_size": [32],
        f"{proto_name}__epochs": [50],
        f"{proto_name}__patience": [15],
        f"{proto_name}__temperature": [0.1],
        f"{proto_name}__supcon_weight": [0.1],
        f"{proto_name}__use_phase_attention": [True],
        f"{proto_name}__modality_dropout_prob": [0.3],
        f"{proto_name}__use_attention_prototypes": [True],
        f"{proto_name}__n_way": [9],
        f"{proto_name}__n_support": [60],
        f"{proto_name}__n_query": [50],
        f"{proto_name}__use_mixup": [False],
        f"{proto_name}__use_time_mask": [False],
        f"{proto_name}__use_gaussian_noise": [False],
    }
    if mode == "single": param_space[f"{proto_name}__target"] = [single_target]
    else:
        param_space[f"{proto_name}__primary_target"] = [primary_target]
        param_space[f"{proto_name}__sub_heads"] = [tuple(multi_heads)]
        param_space[f"{proto_name}__uncertainty_weighting"] = [True]

print(f"✅ Parameter space configured. Total parameters: {len(param_space)}")


✅ Parameter space configured. Total parameters: 40


 # 5. Search Execution

In [ ]:
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        estimator=pipeline, search_spaces=param_space, n_iter=n_iter,
        scoring=competition_scorer, cv=cv_object, n_jobs=n_jobs,
        random_state=random_state, verbose=verbose, refit=True,
        return_train_score=True, error_score=error_score_constant
    )
else:
    search = GridSearchCV(
        estimator=pipeline, param_grid=param_space, scoring=competition_scorer,
        cv=cv_object, n_jobs=n_jobs, verbose=verbose, refit=True,
        return_train_score=True, error_score=error_score_constant,
    )

print(f"🚀 Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\n🏆 Best CV competition score: {search.best_score_:.4f}")
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

best_model = search.best_estimator_


🚀 Running grid search...
Fitting 1 folds for each of 7 candidates, totalling 7 fits


 # 6. Save Results & Holdout Evaluation

In [ ]:
cv_results_df = pd.DataFrame(search.cv_results_)
cv_results_df['mode'] = mode
cv_results_df['search_mode'] = search_mode
cv_results_df['timestamp'] = timestamp
results_path = results_dir / f"cv_results_{mode}_{search_mode}_{timestamp}.csv"
cv_results_df.to_csv(results_path, index=False)
print(f"✅ CV results saved to: {results_path}")

y_test_true = hold_out_df[["sequence_id", "is_target", target_col]].copy()
y_pred = best_model.predict(X_test)

holdout_eval = evaluate_holdout(y_test_true, y_pred, target_col=primary_target if mode == "multi" else target_col)
print(f"\n🎯 Holdout Competition Score: {holdout_eval['competition_score']:.4f}")

holdout_results_path = results_dir / f"holdout_predictions_{timestamp}.csv"
holdout_eval["results_df"].to_csv(holdout_results_path, index=False)
print(f"✅ Holdout predictions saved to: {holdout_results_path}")